<a href="https://colab.research.google.com/github/lekshmi29-lx/DSA_B7_Note-1/blob/main/Data_Ingestion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Data Ingestion**: Each Code explained

1.  **Import necessary libraries**: Imports `pandas` for data manipulation, `time` for introducing delays, and `Audio`, `display` from `IPython.display` for displaying audio alerts.
2.  **Load data from URL**: Reads a CSV file directly from a GitHub URL into a pandas DataFrame.
3.  **Inspect column names**: Displays the original column names.
4.  **Clean column names**: Removes leading/trailing spaces and replaces spaces and forward slashes with underscores in column names for easier access.
5.  **Check data types and non-null values**: Provides information about the data types of each column and the number of non-null entries.
6.  **Convert data types**: Converts the 'Date_Time' column to datetime objects and 'Temp_C' to numeric type.
7.  **Check for missing values**: Confirms there are no missing values in the dataset.
8.  **Implement real-time simulation with alert**: Defines a function `alert_high_temp` to check for temperatures above 3 degrees Celsius and trigger an audio alert. It also defines `stream_with_alert` to simulate processing the data row by row with a delay and stop when a high temperature is detected.

**Apache Kafka**

Apache Kafka is a distributed event streaming platform.
That means it helps applications send, receive, store, and process real-time data streams reliably and at scale.

Kafka ecosystem explained with Uber/Ola example

1. Producer

Who sends data to Kafka topics.

Uber/Ola Example:

Rider app → produces a ride-request event

Driver app → produces driver-location updates every few seconds

Payment service → produces payment-completed events

2. Topic

A logical channel (like a queue) where producers publish messages.

Uber/Ola Example Topics:

ride-requests

driver-locations

payments

notifications

fraud-events

3. Partition

Each topic is split into partitions to allow parallelism & scalability.

Uber/Ola Example:

driver-locations might have 10 partitions.

Partitioning can be based on driver_id or city_id (so all events for a driver go to the same partition).

Benefits: multiple consumers can read in parallel.

4. Consumer

Services or apps that subscribe to topics and process data.

Uber/Ola Example Consumers:

Matching Service → consumes from ride-requests + driver-locations.

Pricing Service → consumes from ride-requests (to calculate surge).

Fraud Detection ML → consumes from payments + ride-events.

Data Warehouse Loader → consumes everything to store in Hadoop/Snowflake for offline analytics.

5. Broker

Kafka server that stores data and serves producers/consumers.

Cluster has multiple brokers (say 10, 50, or 100).

Each broker stores some partitions of topics.

Uber/Ola Example:

Broker 1 stores partitions 0–2 of driver-locations.

Broker 2 stores partitions 3–5 of driver-locations.

This way, Kafka can scale horizontally.

6. Zookeeper (older Kafka versions; now Kafka is moving to KRaft mode)

Keeps track of cluster metadata & coordination:

Which broker is alive.

Who is leader for each partition.

Keeps configs.

Uber/Ola Example: ensures when a broker storing ride requests fails, another broker takes over.

7. Offset

A pointer/position of a message inside a partition.

Each message has an offset = unique ID within that partition.

Consumers use offsets to know where they left off.

Uber/Ola Example:

Matching service consumed messages up to offset 12345 in partition 2 of ride-requests.

If it crashes, it can restart from that offset (no lost rides).

In [ ]:
import pandas as pd
import time
from IPython.display import Audio,display

In [ ]:
url = ("https://raw.githubusercontent.com/velicki/Weather_Data_Analysis_Project/main/Weather_Data.csv")
data = pd.read_csv(url)
data

,Date/Time,Temp_C,Dew Point Temp_C,Rel Hum_%,Wind Speed_km/h,Visibility_km,Press_kPa,Weather
0,1/1/2012 0:00,-1.8,-3.9,86,4,8.0,101.24,Fog
1,1/1/2012 1:00,-1.8,-3.7,87,4,8.0,101.24,Fog
2,1/1/2012 2:00,-1.8,-3.4,89,7,4.0,101.26,"Freezing Drizzle,Fog"
3,1/1/2012 3:00,-1.5,-3.2,88,6,4.0,101.27,"Freezing Drizzle,Fog"
4,1/1/2012 4:00,-1.5,-3.3,88,7,4.8,101.23,Fog
...,...,...,...,...,...,...,...,...
8779,12/31/2012 19:00,0.1,-2.7,81,30,9.7,100.13,Snow
8780,12/31/2012 20:00,0.2,-2.4,83,24,9.7,100.03,Snow
8781,12/31/2012 21:00,-0.5,-1.5,93,28,4.8,99.95,Snow
8782,12/31/2012 22:00,-0.2,-1.8,89,28,9.7,99.91,Snow


In [ ]:
data.columns

Index(['Date/Time', 'Temp_C', 'Dew Point Temp_C', 'Rel Hum_%',
       'Wind Speed_km/h', 'Visibility_km', 'Press_kPa', 'Weather'],
      dtype='object')

In [ ]:
data.columns = data.columns.str.strip().str.replace(' ','_').str.replace('/','_')
data.columns

Index(['Date_Time', 'Temp_C', 'Dew_Point_Temp_C', 'Rel_Hum_%',
       'Wind_Speed_km_h', 'Visibility_km', 'Press_kPa', 'Weather'],
      dtype='object')

In [ ]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8784 entries, 0 to 8783
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Date_Time         8784 non-null   object 
 1   Temp_C            8784 non-null   float64
 2   Dew_Point_Temp_C  8784 non-null   float64
 3   Rel_Hum_%         8784 non-null   int64  
 4   Wind_Speed_km_h   8784 non-null   int64  
 5   Visibility_km     8784 non-null   float64
 6   Press_kPa         8784 non-null   float64
 7   Weather           8784 non-null   object 
dtypes: float64(4), int64(2), object(2)
memory usage: 549.1+ KB


In [ ]:
data['Date_Time'] = pd.to_datetime(data['Date_Time'])
data['Temp_C'] = pd.to_numeric(data['Temp_C'])
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8784 entries, 0 to 8783
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Date_Time         8784 non-null   datetime64[ns]
 1   Temp_C            8784 non-null   float64       
 2   Dew_Point_Temp_C  8784 non-null   float64       
 3   Rel_Hum_%         8784 non-null   int64         
 4   Wind_Speed_km_h   8784 non-null   int64         
 5   Visibility_km     8784 non-null   float64       
 6   Press_kPa         8784 non-null   float64       
 7   Weather           8784 non-null   object        
dtypes: datetime64[ns](1), float64(4), int64(2), object(1)
memory usage: 549.1+ KB


In [ ]:
data.isna().sum()

,0
Date_Time,0
Temp_C,0
Dew_Point_Temp_C,0
Rel_Hum_%,0
Wind_Speed_km_h,0
Visibility_km,0
Press_kPa,0
Weather,0


In [ ]:
def alert_high_temp(row):
  if(row['Temp_C']>3):
    print(f"Alert!!!Temperature is too high: {row['Temp_C']} celsius")
    display(Audio("/content/new-notification-026-380249.mp3",autoplay = True))
    return True
  else:
    print(f"Temperature is normal: {row['Temp_C']} celsius")
    return False


def stream_with_alert(data,delay= 2):
  for _,row in data.iterrows():
    print(f"{row['Date_Time']} - Temperature : {row['Temp_C']} Celsius")
    if alert_high_temp(row):
      print("High Temperature alert triggered.Stopping stream processing")
      break
    time.sleep(delay)

stream_with_alert(data)

2012-01-01 00:00:00 - Temperature : -1.8 Celsius
Temperature is normal: -1.8 celsius
2012-01-01 01:00:00 - Temperature : -1.8 Celsius
Temperature is normal: -1.8 celsius
2012-01-01 02:00:00 - Temperature : -1.8 Celsius
Temperature is normal: -1.8 celsius
2012-01-01 03:00:00 - Temperature : -1.5 Celsius
Temperature is normal: -1.5 celsius
2012-01-01 04:00:00 - Temperature : -1.5 Celsius
Temperature is normal: -1.5 celsius
2012-01-01 05:00:00 - Temperature : -1.4 Celsius
Temperature is normal: -1.4 celsius
2012-01-01 06:00:00 - Temperature : -1.5 Celsius
Temperature is normal: -1.5 celsius
2012-01-01 07:00:00 - Temperature : -1.4 Celsius
Temperature is normal: -1.4 celsius
2012-01-01 08:00:00 - Temperature : -1.4 Celsius
Temperature is normal: -1.4 celsius
2012-01-01 09:00:00 - Temperature : -1.3 Celsius
Temperature is normal: -1.3 celsius
2012-01-01 10:00:00 - Temperature : -1.0 Celsius
Temperature is normal: -1.0 celsius
2012-01-01 11:00:00 - Temperature : -0.5 Celsius
Temperature is n

High Temperature alert triggered.Stopping stream processing
